In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os


load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, built_index

documents = load_faq_data()
index = built_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
CONTEXT:
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag_pipeline("How do I run Olama locally?")

'The context does not provide instructions on how to run Ollama locally. It only mentions that you can "Serve a model locally with [Ollama](https://ollama.com/), [vLLM](https://github.com/vllm-project/vllm), LM Studio, or anything else" as an alternative to using a hosted LLM provider, but it does not provide specific steps or instructions on how to do so.'

In [5]:
assistant.rag_pipeline("How do I run Ollama locally?")

'To run Ollama locally, you need to follow these steps:\n\n1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n   - For **macOS**: Download the `.pkg` and install it.\n   - For **Windows**: Download the `.msi` and install it.\n   - For **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.\n\n2. Once installed, open a terminal and type `ollama run llama3`. This command will:\n   - Download the LLaMA 3 model (~4GB).\n   - Start the model locally.\n   - Open a chat-like interface where you can type questions.\n\n3. To test the Ollama local server, run `curl http://localhost:11434`. You should receive a response similar to `{"models": [...]}`.\n\n4. Then, install the Python client with `pip install ollama`.\n\n5. You can use the following minimal Python example:\n   ```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [7]:
search_tool ={
            "type": "function",
            "name": "search",
            "description":"search in the FAQ database for entries match the given query",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query text to look up in the course FAQ."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }




In [8]:

instructions="""You are a helpful assistant with access to tools.
    You must call tools using valid JSON ONLY.
    You're task is to Answer the QUESTION based on the CONTEXT from the FAQ database.
    """

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]


In [10]:
response = openai_client.responses.create(
    model="openai/gpt-oss-20b",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
)

In [11]:
import json
call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kwryk8bqejpsjzq34w9rnm3r', summary=[], type='reasoning', content=[Content(text='The user says: "I just discovered the course. Can I join it?" We need to answer based on FAQ database. Likely there is an FAQ entry about enrollment. Use search function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join the course I just discovered course join it"}', call_id='fc_1989e53c-30dc-44d5-926b-b8b4a92a22d2', name='search', type='function_call', id='fc_1989e53c-30dc-44d5-926b-b8b4a92a22d2', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'fc_1989e53c-30dc-44d5-926b-b8b4a92a22d2',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "an

In [14]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    instructions=instructions,
    tools=[search_tool],
).output_text

In [15]:
response

'Yes, you can join the course, even if it has already started. You can still register for the course and take part in it, but you might miss some of the homework submissions. To be eligible for a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline.'

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [17]:
def make_call(call):
    args = json.loads(call.arguments)
    if call.name == "search":
        results = search(**args)
    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [18]:
question = "I just discovered the course. Can I join it?"
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)
messages.extend(response)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"joining the course deadline"}


In [19]:
def agent_loop(instructions, question, model="llama-3.3-70b-versatile")->str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ''
    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="llama-3.3-70b-versatile",
            input=messages,
            tools=[search_tool],
        )
        messages.extend(response.output)
        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it+=1
        if not has_function_calls:
            break
    return last_answer



In [20]:
answer = agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local installation instructions"}
function_call: search {"query":"running Olama on local machine"}
iteration #2...
function_call: search {"query":"installing Olama on local machine"}
function_call: search {"query":"Olama installation guide"}
function_call: search {"query":"running Olama locally"}
function_call: search {"query":"Olama local setup"}
iteration #3...
ASSISTANT:
Unfortunately, I couldn't find any information on how to run Olama locally. If you have any more questions or need further assistance, feel free to ask.


In [21]:
answer

"Unfortunately, I couldn't find any information on how to run Olama locally. If you have any more questions or need further assistance, feel free to ask."

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"course enrollment deadline"}
function_call: search {"query":"joining course after start date"}
function_call: search {"query":"late course registration"}
iteration #2...
ASSISTANT:
Yes, you can still join the course after it has started. While you might miss the first couple of homework assignments, they are not required for the certificate. To earn the certificate, you need to pass a project and evaluate 3 of your peers' projects, which starts later in the course. 

Are there any other areas of the course you'd like to explore or have questions about?


"Yes, you can still join the course after it has started. While you might miss the first couple of homework assignments, they are not required for the certificate. To earn the certificate, you need to pass a project and evaluate 3 of your peers' projects, which starts later in the course. \n\nAre there any other areas of the course you'd like to explore or have questions about?"

In [24]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [51]:
agent_tool = Tools()
agent_tool.add_tool(search)

In [52]:
agent_tool.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [53]:
chat_interface = IPythonChatInterface()
callback =  DisplayingRunnerCallback(chat_interface)

In [63]:
runner = OpenAIResponsesRunner(
    developer_prompt=instructions,
    chat_interface=chat_interface,
    tools=agent_tool,
    llm_client=OpenAIClient(
        client = openai_client,
        model="openai/gpt-oss-20b"
    )
)

In [64]:
result = runner.loop(
    "How do I run Olama locally?, i have a big GPU",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


/home/mohamed-farrag/Learning/Courses/LLM Zoomcamp 2026/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-20b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [65]:
result.last_message

'### Quick‑start: Running Ollama locally on a machine with a big GPU\n\n> **TL;DR**  \n> 1. Install Ollama (or Docker‑based).  \n> 2. Pull the model you want (`ollama pull llama3`, `ollama pull phi3`, etc.).  \n> 3. Start the local server (`ollama serve` or `ollama run <model>`).  \n> 4. Verify the GPU is being used (`nvidia-smi`).  \n> 5. (Optional) Use the Python client or a simple HTTP API to query the model.\n\nBelow you’ll find step‑by‑step instructions for the three major OS families, plus a Docker‑based route if you prefer container isolation. Because you have a *big GPU*, you can also choose a larger model that fits your memory.\n\n---\n\n## 1️⃣ Install Ollama on the host (native)\n\n| OS | Command | Notes |\n|---|---|---|\n| **Linux (x86‑64)** | ```bash\\ncurl -fsSL https://ollama.com/install.sh | sh\\n``` | The script installs the binary in `/usr/local/bin` and sets up the system service. |\n| **macOS (Apple Silicon or Intel)** | ```bash\\nbrew install ollama   # via Homebrew

In [66]:
result = runner.loop(
    prompt = "How to run other models?",
    previous_messages=result.all_messages,
    callback=callback,
)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-20b` in organization `org_01kvw2kpzsfrv8gaagb9g013gg` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8408, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [62]:
runner.run()

-> Response received


-> Response received


/home/mohamed-farrag/Learning/Courses/LLM Zoomcamp 2026/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


-> Response received


/home/mohamed-farrag/Learning/Courses/LLM Zoomcamp 2026/.venv/lib/python3.13/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.3-70b-versatile'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: attempted to call tool \'search={"query": "Ollama in docker"}\' which was not in request.tools', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search={"query": "Ollama in docker"}></function>'}}